<h1 style="color:Red; font-weight:bold;">2.2 Tokenizing Text</h1>   

In [4]:
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt")

file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x26c506b7b10>)

In [5]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [6]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [7]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [8]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [9]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [10]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [11]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


<h1 style="color:blue; font-weight:bold;">2.3 Tokens to Token IDs</h1>   

In [12]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [13]:
vocab = {token:integer for integer,token in enumerate(all_words)}
# i have not included the code of printing the first few entries to check if vocab is correct or not.

In [14]:
class SimpleTokenizerV1:
    
    def __init__(self, vocab):
        # vocab is a dictionary like {"hello": 0, "world": 1}
        # str_to_int: word → number (directly from vocab)
        self.str_to_int = vocab
        
        # int_to_str: number → word (reverse of vocab)
        # we flip the key-value pairs using dictionary comprehension
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        # Step 1: Split text into tokens (words + punctuation)
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        
        # Step 2: Remove empty strings and extra whitespace
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        
        # Step 3: Convert each token to its number using str_to_int
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        # Step 1: Convert each number back to its token using int_to_str
        # Step 2: Join all tokens with a space
        text = " ".join([self.int_to_str[i] for i in ids])
        
        # Step 3: Fix spacing before punctuation
        # e.g. "Hello , world" → "Hello, world"
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)
text = """Hi! My name is Annant Pathak. I'm a second Year B.Tech student in IIT Hyderabad"""
ids = tokenizer.encode(text)
print(ids)
# Keyerror Hi -- It means the keyword hi was not in our vocab, so this normal tokenizer can't encode this as it doesn't
# Have the given ID for this text. Thats why we need large datasets to train LLMs. or other things like
# <unk>, <end of text>

KeyError: 'Hi'

In [16]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
 Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [17]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


## 2.4 Adding special context tokens

In [21]:
# We're adding two special tokens end of text and unk
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}
print(len(vocab.items()))

1132


In [22]:
# Concept of tokenizer 2 is similar to tokenizer 1, we're just adding the logic of handling unknown words.
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int
                            else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [23]:
text1 = "Hi i'm Annant Pathak."
text2 = "IIT hyderabad is sixth in NIRF."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hi i'm Annant Pathak. <|endoftext|> IIT hyderabad is sixth in NIRF.


In [ ]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))
#always the unknown text is kept at last of vocab, so you can see so many id as 1131 it means they are unknown.

[1131, 1131, 2, 1131, 1131, 1131, 7, 1130, 1131, 1131, 584, 1131, 568, 1131, 7]


In [25]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|> <|unk|>' <|unk|> <|unk|> <|unk|>. <|endoftext|> <|unk|> <|unk|> is <|unk|> in <|unk|>.


## 2.5 Byte Pair Encoding (BPE)

In [27]:
# BPE is complex so we won't code it. We'll import it from Tiktoken library.
!pip install tiktoken
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


tiktoken version: 0.12.0


In [34]:
tokenizer = tiktoken.get_encoding("gpt2")

ConnectTimeout: HTTPSConnectionPool(host='openaipublic.blob.core.windows.net', port=443): Max retries exceeded with url: /gpt-2/encodings/main/vocab.bpe (Caused by ConnectTimeoutError(<HTTPSConnection(host='openaipublic.blob.core.windows.net', port=443) at 0x26c513c2490>, 'Connection to openaipublic.blob.core.windows.net timed out. (connect timeout=None)'))

In [30]:
text = (
 "Hi, i'm Annant Pathak <|endoftext|> IIT hyderabad is the sixth in NIRF."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

TypeError: SimpleTokenizerV2.encode() got an unexpected keyword argument 'allowed_special'

In [32]:
strings = tokenizer.decode(integers)
print(strings)

NameError: name 'integers' is not defined